<a href="https://colab.research.google.com/github/ven123-commits/KANAD_LAW_GARDEN/blob/main/KANAD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [39]:
!pip install google-generativeai pdfplumber -q

In [57]:
from google.colab import userdata
import google.generativeai as genai

genai.configure(api_key=userdata.get('GEMINI_API_KEY'))
model = genai.GenerativeModel("gemini-3.5-flash-lite")
#model = genai.GenerativeModel("gemini-flash-latest")

In [58]:
import google.generativeai as genai

for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/gemini-3.7-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.6-preview
models/gemini-robotics-er-2-preview
models/gemini-2.5-computer-use-p

In [53]:
from google.colab import files
uploaded = files.upload()


Saving disturbedareasact.pdf to disturbedareasact (1).pdf
Saving increaseofstampduty.pdf to increaseofstampduty (1).pdf
Saving landimprovementschemes.pdf to landimprovementschemes (1).pdf
Saving landrequisitionact.pdf to landrequisitionact (1).pdf
Saving mamlatdarscourtsact.pdf to mamlatdarscourtsact (1).pdf
Saving mergedstateslawsact.pdf to mergedstateslawsact (1).pdf
Saving preventionoffragmentationact.pdf to preventionoffragmentationact (1).pdf
Saving prohibitionofleases.pdf to prohibitionofleases (1).pdf
Saving requisitionedproperty.pdf to requisitionedproperty (1).pdf
Saving revenue_jurisdiction_act1876.pdf to revenue_jurisdiction_act1876 (1).pdf


In [59]:
import pdfplumber

def extract_text(pdf_path):
    full_text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                full_text += page_text + "\n"
    return full_text.strip()

In [60]:
import json

def process_document(text):
    prompt = f"""You are analyzing an Indian government or legal document.
Read the text below and return ONLY a JSON object, nothing else — no markdown formatting, no explanation, no code fences.

The JSON must have exactly this shape:
{{
  "doc_type": "Act" or "GR" or "Notification" or "Circular" or "Order",
  "department": "the government department this belongs to, e.g. Education, Revenue, Health",
  "title": "the official title of the document",
  "summary_en": "a 3-4 sentence plain-English summary of what this document says",
  "referenced_acts": ["list of any other Act names mentioned in this document, empty list if none"]
}}

Document text:
{text[:8000]}
"""
    response = model.generate_content(prompt, generation_config={"temperature": 0.2})
    raw_output = response.text.strip()

    if raw_output.startswith("```"):
        raw_output = raw_output.strip("`")
        raw_output = raw_output.replace("json", "", 1).strip()

    return json.loads(raw_output)

In [61]:
import time
pdf_path = "disturbedareasact.pdf"  # or your Drive path from Step 3

text = extract_text(pdf_path)
print("--- Extracted text (first 500 chars) ---")
print(text[:500])

result = process_document(text)
print("\n--- Structured JSON result ---")
print(result)

time.sleep(3)  # add after each process_document() call in your batch loop

--- Extracted text (first 500 chars) ---
The Gujarat Prohibition of Transfer of Immovable Property and Provision for
Protection of Tenants from Eviction from Premises in Disturbed Areas Act, 1991.
GOVERNMENT OF GUJARAT
LEGISLATIVE AND PARLIAMENTARY AFFAIRS DEPARTMENT
Gujarat Act No. 12 of 1991
The Gujarat Prohibition of Transfer of
Immovable Property and Provision for
Protection of Tenants from Eviction from
Premises in Disturbed Areas Act, 1991.
(As modified upto the 31st December, 2005)
1 of 7
The Gujarat Prohibition of Transfer of I

--- Structured JSON result ---
{'doc_type': 'Act', 'department': 'Legislative and Parliamentary Affairs Department', 'title': 'The Gujarat Prohibition of Transfer of Immovable Property and Provision for Protection of Tenants from Eviction from Premises in Disturbed Areas Act, 1991', 'summary_en': "This Act allows the Government of Gujarat to declare areas affected by riots or mob violence as 'disturbed areas' for a specified period. During these periods

In [62]:
for m in genai.list_models():
    if 'embedContent' in m.supported_generation_methods:
        print(m.name)

models/gemini-embedding-001
models/gemini-embedding-2-preview
models/gemini-embedding-2


In [63]:
def get_embedding(text):
    result = genai.embed_content(
        model="models/gemini-embedding-001",
        content=text,
        task_type="retrieval_document"
    )
    return result['embedding']

def get_query_embedding(query):
    result = genai.embed_content(
        model="models/gemini-embedding-001",
        content=query,
        task_type="retrieval_query"
    )
    return result['embedding']

In [64]:
# Process a small batch of documents (2-3 for testing)
pdf_paths = ["disturbedareasact.pdf", "landimprovementschemes.pdf", "mamlatdarscourtsact.pdf"]  # add your actual filenames

processed_docs = []
for path in pdf_paths:
    text = extract_text(path)
    summary_data = process_document(text)
    embedding = get_embedding(text[:8000])  # embed the same truncated text you summarized

    processed_docs.append({
        "path": path,
        **summary_data,
        "embedding": embedding
    })

print(f"Processed {len(processed_docs)} documents")

Processed 3 documents


In [65]:
import numpy as np

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def search(query, documents, top_k=3):
    query_emb = get_query_embedding(query)

    scored = []
    for doc in documents:
        score = cosine_similarity(query_emb, doc["embedding"])
        scored.append((score, doc))

    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[:top_k]

In [66]:
results = search("stamp duty on property", processed_docs)

for score, doc in results:
    print(f"Score: {score:.3f} | Title: {doc.get('title')} | Type: {doc.get('doc_type')}")

Score: 0.614 | Title: The Gujarat Prohibition of Transfer of Immovable Property and Provision for Protection of Tenants from Eviction from Premises in Disturbed Areas Act, 1991 | Type: Act
Score: 0.576 | Title: The Mamlatdars' Courts Act, 1906 | Type: Act
Score: 0.565 | Title: The Gujarat Land Improvement Schemes Act, 1942 | Type: Act


In [67]:
results = search("land acquisition by government", processed_docs)

for score, doc in results:
    print(f"Score: {score:.3f} | Title: {doc.get('title')} | Type: {doc.get('doc_type')}")

Score: 0.649 | Title: The Gujarat Land Improvement Schemes Act, 1942 | Type: Act
Score: 0.646 | Title: The Gujarat Prohibition of Transfer of Immovable Property and Provision for Protection of Tenants from Eviction from Premises in Disturbed Areas Act, 1991 | Type: Act
Score: 0.618 | Title: The Mamlatdars' Courts Act, 1906 | Type: Act


In [68]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [69]:
import os

folder_path = "/content/drive/MyDrive/KANAD docs (acts and gr)/K.A.NA.D Hackathon/acts/revenue"  # update this
pdf_files = [f for f in os.listdir(folder_path) if f.endswith(".pdf")]

print(f"Found {len(pdf_files)} PDFs")

Found 16 PDFs


In [70]:
processed_docs = []
failed_docs = []

for filename in pdf_files:
    full_path = os.path.join(folder_path, filename)
    try:
        text = extract_text(full_path)
        if not text or len(text) < 50:
            failed_docs.append((filename, "no extractable text — likely scanned image"))
            continue

        summary_data = process_document(text)
        embedding = get_embedding(text[:8000])

        processed_docs.append({
            "filename": filename,
            **summary_data,
            "embedding": embedding
        })
        print(f"✓ {filename}")
    except Exception as e:
        failed_docs.append((filename, str(e)))
        print(f"✗ {filename} — {e}")

print(f"\nProcessed: {len(processed_docs)} | Failed: {len(failed_docs)}")

import time
time.sleep(3)  # add after each process_document() call in your batch loop

✓ revenue_jurisdiction_act1876.pdf
✓ mamlatdarscourtsact.pdf
✓ landimprovementschemes.pdf
✓ preventionoffragmentationact.pdf
✓ landrequisitionact.pdf
✓ saurashtra_gharkhed_ordinance_1949.pdf
✓ mergedstateslawsact.pdf
✓ prohibitionofleases.pdf
✓ revenuetribunalact.pdf
✓ stamp_act.pdf
✓ requisitionedproperty.pdf
✓ 001_title-h-2003_(the_bombay_tenancy_and_agricultural_lands_rule_1958).pdf
✓ increaseofstampduty.pdf
✓ The Saurashtra Esttates Acquisition Act-1952.pdf
✓ saurashtralandreforms.pdf
✓ disturbedareasact.pdf

Processed: 16 | Failed: 0


In [71]:
import json

with open('/content/drive/MyDrive/KANAD docs (acts and gr)/processed_results.json', 'w') as f:
    json.dump(processed_docs, f)

print(f"Saved {len(processed_docs)} processed documents")

Saved 16 processed documents


In [79]:
results = search("property transfer restrictions", processed_docs)
for score, doc in results:
    print(f"Score: {score:.3f} | Title: {doc.get('title')} | Type: {doc.get('doc_type')}")

Score: 0.690 | Title: The Gujarat Prohibition of Transfer of Immovable Property and Provision for Protection of Tenants from Eviction from Premises in Disturbed Areas Act, 1991 | Type: Act
Score: 0.630 | Title: The Bombay Requisitioned Property (Continuance of Powers) (Saurashtra Area) Act, 1958 | Type: Act
Score: 0.620 | Title: The Saurashtra Prohibition of Leases of Agricultural Lands Act, 1953 | Type: Act


In [76]:
from fastapi import FastAPI
from db import search_by_embedding  # Person 2's function, queries Supabase

app = FastAPI()

@app.get("/search")
def search_endpoint(q: str):
    query_embedding = get_query_embedding(q)  # your function from today
    results = search_by_embedding(query_embedding, top_k=5)  # Person 2's function
    return {"query": q, "results": results}

ModuleNotFoundError: No module named 'db'

In [77]:
def process_document(text):
    prompt = f"""You are analyzing an Indian government or legal document.
Read the text below and return ONLY a JSON object, nothing else — no markdown formatting, no explanation, no code fences.

The JSON must have exactly this shape:
{{
  "doc_type": "Act" or "GR" or "Notification" or "Judgment" or "Scheme",
  "department": "the government department this belongs to, e.g. Education, Revenue, Health",
  "title": "the official title of the document",
  "summary_en": "a 3-4 sentence plain-English summary of what this document says",
  "summary_hi": "the same summary translated into natural, fluent Hindi (Devanagari script)",
  "summary_gu": "the same summary translated into natural, fluent Gujarati (Gujarati script)",
  "referenced_acts": ["list of any other Act names mentioned in this document, empty list if none"]
}}

Document text:
{text[:8000]}
"""
    response = model.generate_content(prompt, generation_config={"temperature": 0.2})
    raw_output = response.text.strip()

    if raw_output.startswith("```"):
        raw_output = raw_output.strip("`")
        raw_output = raw_output.replace("json", "", 1).strip()

    return json.loads(raw_output)

import time
time.sleep(3)  # add after each process_document() call in your batch loop

In [78]:
test_doc = processed_docs[0]  # or pick a specific filename
text = extract_text(os.path.join(folder_path, test_doc["filename"]))
result = process_document(text)

print("English:", result["summary_en"])
print("\nHindi:", result["summary_hi"])
print("\nGujarati:", result["summary_gu"])

English: The Bombay Revenue Jurisdiction Act, 1876, is an enactment designed to limit the jurisdiction of Civil Courts in matters relating to land revenue across certain regions of the former Bombay Presidency. It explicitly bars civil courts from entertaining certain suits and claims against the government concerning property, hereditary offices, village servants, and revenue administration matters. The Act provides definitions for essential terms like 'land', 'land-revenue', and 'revenue officer' to maintain clarity in revenue-related legal proceedings. Furthermore, it outlines specific conditions and exceptions under which revenue matters may or may not be challenged or referred to higher judicial authorities.

Hindi: बॉम्बे राजस्व अधिकार क्षेत्र अधिनियम, 1876 एक ऐसा अधिनियम है जिसे पूर्व बॉम्बे प्रेसीडेंसी के कुछ क्षेत्रों में भूमि राजस्व से संबंधित मामलों में सिविल अदालतों के अधिकार क्षेत्र को सीमित करने के लिए बनाया गया था। यह स्पष्ट रूप से सिविल अदालतों को संपत्ति, आनुवंशिक कार्